# Context safe category classification

Phi-4-mini QLoRA, architectures A and C; training seed 42, split seed 123.


In [ ]:
from pathlib import Path
import os, json, gc, csv, hashlib, math, random
from collections import Counter

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'pyproject.toml').is_file() and (p / 'data/forensic_atomic.csv').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Run this notebook from inside the forensic_atomic repository.')
import sys
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
EXP = ROOT / 'models/phi4_mini/classification/context_safe'
SPLITS = EXP / 'splits'
OUTPUTS = EXP / 'adapters'
RESULTS = ROOT / 'results/phi4_mini/classification/context_safe'
MODEL_ID = 'microsoft/Phi-4-mini-instruct'
SEED = 42
MAX_ITEMS_PER_RELATION = 0
RELATIONS = ['xIntent','xNeed','xAttr','xEffect','xReact','xWant',
             'oEffect','oReact','oWant','isAfter','HasSubEvent','isBefore',
             'HinderedBy','Causes','xReason','isFilledBy']

def sha256_file(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda:f.read(1024*1024), b''):
            h.update(block)
    return h.hexdigest()

def read_jsonl(path):
    with Path(path).open(encoding='utf-8') as f:
        return [json.loads(s) for s in f if s.strip()]

assert (SPLITS/'manifest.json').exists(), 'Run 01_prepare.ipynb first (CPU)'
manifest = json.loads((SPLITS/'manifest.json').read_text(encoding='utf-8'))
for name, checksum in manifest['file_sha256'].items():
    assert sha256_file(SPLITS/name) == checksum, f'File changed: {name}'
train_rows = read_jsonl(SPLITS/'train.jsonl')
val_rows = read_jsonl(SPLITS/'val.jsonl')
test_rows = read_jsonl(SPLITS/'test.jsonl')
taxonomy = json.loads((SPLITS/'taxonomy.json').read_text(encoding='utf-8'))
OUTPUTS.mkdir(parents=True,exist_ok=True)
RESULTS.mkdir(parents=True,exist_ok=True)
print('Experiment:', EXP)
print('Train / Val / Test:', len(train_rows), len(val_rows), len(test_rows))


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from datasets import Dataset
from peft import PeftModel, LoraConfig, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
print('PyTorch:',torch.__version__)
print('CUDA:',torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Use a GPU kernel; training on CPU is unsupported.')
print('GPU:',torch.cuda.get_device_name(0))
print('VRAM GiB:',round(torch.cuda.get_device_properties(0).total_memory/1024**3,2))
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID,trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
TAXONOMY_TEXT='\n'.join(f'- {cat}: {", ".join(sub)}' for cat, sub in sorted(taxonomy.items()))


## Classification prompt


In [ ]:
def make_prompt(record, architecture):
    assert architecture in ('A','C')
    system = f"""You are a forensic analysis expert.

Your task is to classify each provided forensic scenario by assigning exactly one crime category and one corresponding crime subcategory.

### Classification Taxonomy ###

{TAXONOMY_TEXT}

### Classification Rules ###

1. Analyze the event and all the information explicitly provided for the scenario.
2. Assign exactly one crime category and one corresponding crime subcategory.
3. Use only the categories and subcategories listed in the taxonomy. Do not create additional labels.
4. If multiple classifications could apply, select the one that best represents the main criminal conduct.
5. Do not invent facts, intentions or circumstances unsupported by the provided information.
6. Treat any stated intentions, consequences or inferred relations as contextual evidence, not necessarily as established facts.

### Output Format ###

Return ONLY a valid JSON object:

{{
  "crime_category": "category",
  "crime_subcategory": "subcategory"
}}

Do not provide additional explanations.
Do not use markdown."""
    user = f'Event: {record["event"]}\nBrief context: {record["brief_context"]}'
    if architecture == 'C':
        rel_lines = []
        for rel in RELATIONS:
            values = record[rel]
            if MAX_ITEMS_PER_RELATION:
                values = values[:MAX_ITEMS_PER_RELATION]
            rel_lines.append(f'{rel}: {json.dumps(values,ensure_ascii=False)}')
        user += '\nATOMIC forensic relation annotations:\n' + '\n'.join(rel_lines)
    return tokenizer.apply_chat_template(
        [{'role':'system','content':system}, {'role':'user','content':user}],
        tokenize=False, add_generation_prompt=True
    )

def target_text(record):
    return json.dumps({'crime_category':record['crime_category'],
                       'crime_subcategory':record['crime_subcategory']},
                      ensure_ascii=False,separators=(',',':')) + '<|end|>'

print('Prompt A (first 600 characters):\n',make_prompt(train_rows[0],'A')[:600])
print('Prompt C (first 600 characters):\n',make_prompt(train_rows[0],'C')[:600])
print('Target:',target_text(train_rows[0]))


In [ ]:
def check_lengths(arch, max_length=None, train_limit=0, val_limit=0):
    limit = max_length or (3072 if arch=='A' else 4096)
    for name, data, n in [('train',train_rows,train_limit),('val',val_rows,val_limit)]:
        selected = data[:n] if n else data
        sizes = [len(tokenizer(make_prompt(row,arch)+target_text(row),
                               add_special_tokens=False)['input_ids']) for row in selected]
        too_long=sum(size > limit for size in sizes)
        print(f'{arch} {name}: {len(selected)} examples, max={max(sizes)}, limit={limit}, over limit={too_long}')
        if too_long:
            raise ValueError(f'{too_long} sequences {arch} {name} above max_length; stopping to avoid silent truncation')
    return limit


check_lengths('A', train_limit=64,val_limit=16)
check_lengths('C', train_limit=64,val_limit=16)


## QLoRA training


In [ ]:
def valid_checkpoints(folder):
    candidates=[]
    for cp in Path(folder).glob('checkpoint-*'):
        if not cp.is_dir() or not cp.name.split('-')[-1].isdigit():
            continue
        required=['trainer_state.json','optimizer.pt','scheduler.pt','adapter_model.safetensors']
        if all((cp/f).exists() for f in required):
            candidates.append(cp)
    return sorted(candidates,key=lambda p:int(p.name.split('-')[-1]))

def train_clean(architecture, *, full=False, epochs=None, resume=True):
    assert architecture in ('A','C')
    if not torch.cuda.is_available():
        raise RuntimeError('CUDA GPU unavailable')
    name=architecture+'_clean' if full else architecture+'_clean_smoke'
    target=OUTPUTS/name
    if (target/'experiment.json').exists() and (target/'adapter_model.safetensors').exists():
        print('Training already complete; reusing:',target)
        return target
    max_length = 3072 if architecture=='A' else 4096
    lt=0 if full else 64
    lv=0 if full else 16
    ep=(2 if full else 1) if epochs is None else epochs
    check_lengths(architecture,max_length,lt,lv)
    candidates=valid_checkpoints(target) if resume else []
    latest=candidates[-1] if candidates else None
    if target.exists() and any(target.iterdir()) and latest is None:
        raise RuntimeError(f'{target} contains files but no complete checkpoint; inspect the directory before retrying.')
    if latest:
        print('Resuming from:',latest)
    src_t=train_rows if full else train_rows[:lt]
    src_v=val_rows if full else val_rows[:lv]
    make = lambda r: {'prompt':make_prompt(r,architecture),'completion':target_text(r)}
    ds_train=Dataset.from_list([make(r) for r in src_t])
    ds_val=Dataset.from_list([make(r) for r in src_v])
    bf16=torch.cuda.is_bf16_supported()
    dtype=torch.bfloat16 if bf16 else torch.float16
    quant=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type='nf4',
                            bnb_4bit_use_double_quant=True,bnb_4bit_compute_dtype=dtype)
    model=AutoModelForCausalLM.from_pretrained(
        MODEL_ID,quantization_config=quant,torch_dtype=dtype,
        trust_remote_code=True,device_map={'':torch.cuda.current_device()})
    model.config.use_cache=False
    model=prepare_model_for_kbit_training(model)
    lora=LoraConfig(task_type='CAUSAL_LM',r=16,lora_alpha=32,
                    lora_dropout=0.05,target_modules='all-linear')
    config=SFTConfig(
        output_dir=str(target),max_length=max_length,completion_only_loss=True,
        num_train_epochs=ep,per_device_train_batch_size=1,
        per_device_eval_batch_size=1,gradient_accumulation_steps=8,
        learning_rate=2e-4,lr_scheduler_type='cosine',warmup_ratio=0.03,
        logging_strategy='steps',logging_steps=10,logging_first_step=True,
        save_strategy='steps',save_steps=100,save_total_limit=3,
        eval_strategy='epoch',bf16=bf16,fp16=not bf16,
        gradient_checkpointing=True,report_to='none',packing=False,seed=SEED)
    trainer=SFTTrainer(model=model,args=config,train_dataset=ds_train,
                       eval_dataset=ds_val,processing_class=tokenizer,peft_config=lora)
    try:
        trainer.train(resume_from_checkpoint=str(latest) if latest else None)
        trainer.save_model(str(target))
        tokenizer.save_pretrained(str(target))
        (target/'experiment.json').write_text(json.dumps({
            'architecture':architecture,'clean':True,'model':MODEL_ID,
            'full':full,'epochs':ep,'max_length':max_length,'seed_training':SEED,
            'split_sha256':manifest['file_sha256'],'clean_method':manifest['clean_method'],
            'max_items_per_relation':MAX_ITEMS_PER_RELATION
        },indent=2,ensure_ascii=False),encoding='utf-8')
        print('Adapter ready:',target)
    finally:
        del trainer, model
        gc.collect();torch.cuda.empty_cache()
    return target


## Optional smoke runs


## Full training


## Base and adapter inference


In [ ]:
EVAL_SPLIT = 'val'
PRED_LIMIT = 30

def parse_answer(raw):
    s=raw.replace('```json','').replace('```','').replace('<|end|>','').strip()
    try:
        obj,_=json.JSONDecoder().raw_decode(s[s.index('{'):])
        if not isinstance(obj,dict):
            return None,True,False
    except (ValueError,json.JSONDecodeError):
        return None,False,False
    cat,sub=obj.get('crime_category'),obj.get('crime_subcategory')
    return obj,True,isinstance(cat,str) and isinstance(sub,str) and sub in taxonomy.get(cat,[])

@torch.inference_mode()
def predict_clean(architecture, *, fine_tuned=False, split=None, limit=None):
    assert architecture in ('A','C')
    split=split or EVAL_SPLIT
    assert split in ('val','test')
    data={'val':val_rows,'test':test_rows}[split]
    n=PRED_LIMIT if limit is None else limit
    chosen=data[:n] if n else data
    kind='ft' if fine_tuned else 'base'
    name=f'{architecture}_clean_{kind}_{split}_{len(chosen)}'
    out=RESULTS/(name+'.jsonl')
    sigfile=RESULTS/(name+'_signature.json')
    adapter=OUTPUTS/(architecture+'_clean') if fine_tuned else None
    if adapter is not None:
        cfg_file=adapter/'experiment.json'
        if not cfg_file.exists():
            raise FileNotFoundError(f'Incomplete adapter: {adapter}')
        cfg=json.loads(cfg_file.read_text(encoding='utf-8'))
        assert cfg['architecture']==architecture
        assert cfg['split_sha256']==manifest['file_sha256'], 'Adapter trained on different splits'
        assert cfg['max_items_per_relation']==MAX_ITEMS_PER_RELATION
    signature={
        'experiment':'context_truncation_original_seed123','name':name,
        'data_sha256':manifest['file_sha256'][split+'.jsonl'],
        'model':MODEL_ID,'adapter':str(adapter) if adapter else None,
        'max_items_per_relation':MAX_ITEMS_PER_RELATION,
        'prompt_sha256':hashlib.sha256(''.join(make_prompt(row,architecture) for row in chosen).encode('utf-8')).hexdigest(),
        'n':len(chosen)
    }
    if out.exists() or sigfile.exists():
        if not out.exists() or not sigfile.exists():
            raise RuntimeError('Prediction file lacks a signature; inspect it before resuming')
        stored=json.loads(sigfile.read_text(encoding='utf-8'))
        assert signature==stored,'Configuration changed; predictions cannot be reused'
        prior=read_jsonl(out)
        assert len(prior)<=len(chosen)
        for i,r in enumerate(prior):
            expected=chosen[i]
            assert (str(r['id']),r['gold_category'],r['gold_subcategory'])==(
                str(expected['original_row_id']),expected['crime_category'],expected['crime_subcategory']),f'Previous prediction does not match row {i}'
        if len(prior)==len(chosen):
            print('Complete file reused without inference:',out)
            return out
        print(f'Resuming predictions from {len(prior)}/{len(chosen)}')
        begin=len(prior)
    else:
        sigfile.write_text(json.dumps(signature,indent=2),encoding='utf-8')
        out.touch(exist_ok=False)
        begin=0
    bf16=torch.cuda.is_bf16_supported()
    dtype=torch.bfloat16 if bf16 else torch.float16
    quant=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type='nf4',
                            bnb_4bit_use_double_quant=True,bnb_4bit_compute_dtype=dtype)
    model=AutoModelForCausalLM.from_pretrained(
        MODEL_ID,quantization_config=quant,torch_dtype=dtype,
        trust_remote_code=True,device_map={'':torch.cuda.current_device()})
    if adapter:
        model=PeftModel.from_pretrained(model,str(adapter))
    model.eval()
    end_id=tokenizer.convert_tokens_to_ids('<|end|>')
    stop_ids=list({x for x in (tokenizer.eos_token_id,end_id) if isinstance(x,int) and x>=0})
    try:
        with out.open('a',encoding='utf-8') as f:
            for i in range(begin,len(chosen)):
                r=chosen[i]
                inputs=tokenizer(make_prompt(r,architecture),add_special_tokens=False,return_tensors='pt').to('cuda')
                generated=model.generate(**inputs,max_new_tokens=128,do_sample=False,
                                         eos_token_id=stop_ids,pad_token_id=tokenizer.pad_token_id)
                raw=tokenizer.decode(generated[0,inputs['input_ids'].shape[1]:],skip_special_tokens=True)
                obj,json_ok,pair_allowed=parse_answer(raw)
                result={
                    'id':r['original_row_id'],'architecture':architecture,
                    'gold_category':r['crime_category'],'gold_subcategory':r['crime_subcategory'],
                    'pred_category':obj.get('crime_category') if obj else None,
                    'pred_subcategory':obj.get('crime_subcategory') if obj else None,
                    'json_valid':json_ok,'pair_allowed':pair_allowed,'raw_output':raw
                }
                f.write(json.dumps(result,ensure_ascii=False)+'\n')
                f.flush()
                if (i+1)%25==0 or i+1==len(chosen):
                    print(f'{name}: {i+1}/{len(chosen)}')
    finally:
        del model
        gc.collect();torch.cuda.empty_cache()
    print('Results:',out)
    return out


## Prediction runs


## Metrics and comparison


In [ ]:
def macro_f1(y_true,y_pred):
    labels=set(y_true)
    values=[]
    for lab in labels:
        tp=sum(a==lab and b==lab for a,b in zip(y_true,y_pred))
        fp=sum(a!=lab and b==lab for a,b in zip(y_true,y_pred))
        fn=sum(a==lab and b!=lab for a,b in zip(y_true,y_pred))
        values.append(2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0)
    return sum(values)/len(values) if values else 0

def evaluate_clean(architecture, *, fine_tuned=False, split='val',limit=30):
    assert split in ('val','test')
    data={'val':val_rows,'test':test_rows}[split]
    chosen=data[:limit] if limit else data
    name=f'{architecture}_clean_{"ft" if fine_tuned else "base"}_{split}_{len(chosen)}'
    path=RESULTS/(name+'.jsonl')
    assert path.exists(), f'Missing {path}: run the corresponding prediction'
    rows=read_jsonl(path)
    assert len(rows)==len(chosen),f'Incomplete predictions: {len(rows)}/{len(chosen)}'
    gold_c=[r['gold_category'] for r in rows]
    gold_s=[r['gold_subcategory'] for r in rows]
    pred_c=[r['pred_category'] if r['json_valid'] and isinstance(r['pred_category'],str) else '__INVALID__' for r in rows]
    pred_s=[r['pred_subcategory'] if r['json_valid'] and isinstance(r['pred_subcategory'],str) else '__INVALID__' for r in rows]
    n=len(rows)
    metrics={
        'n':n,
        'category_accuracy':sum(a==b for a,b in zip(gold_c,pred_c))/n,
        'category_macro_f1':macro_f1(gold_c,pred_c),
        'subcategory_accuracy':sum(a==b for a,b in zip(gold_s,pred_s))/n,
        'subcategory_macro_f1':macro_f1(gold_s,pred_s),
        'pair_accuracy':sum(r['pair_allowed'] and a==c and b==d for r,a,b,c,d in zip(rows,gold_c,gold_s,pred_c,pred_s))/n,
        'json_valid_rate':sum(bool(r['json_valid']) for r in rows)/n,
        'allowed_pair_rate':sum(bool(r['pair_allowed']) for r in rows)/n
    }
    metric_path=RESULTS/(name+'_metrics.json')
    if metric_path.exists():
        assert json.loads(metric_path.read_text(encoding='utf-8'))==metrics, f'Metrics differ from existing file: {metric_path}'
    else:
        metric_path.write_text(json.dumps(metrics,indent=2),encoding='utf-8')
    print(name,json.dumps(metrics,indent=2))
    return metrics

def compare_clean(*,split='val',limit=30):
    names=[(a,ft) for a in ('A','C') for ft in (False,True)]
    allrows=[]; gold_orders=[]
    data={'val':val_rows,'test':test_rows}[split]
    n=len(data[:limit] if limit else data)
    for arch,ft in names:
        name=f'{arch}_clean_{"ft" if ft else "base"}_{split}_{n}'
        preds=read_jsonl(RESULTS/(name+'.jsonl'))
        gold_orders.append([(r['id'],r['gold_category'],r['gold_subcategory']) for r in preds])
        m=evaluate_clean(arch,fine_tuned=ft,split=split,limit=limit)
        allrows.append({'name':name,**m})
    assert all(x==gold_orders[0] for x in gold_orders),'Predictions use different splits or row order'
    out=RESULTS/(f'comparison_clean_{split}_{n}.csv')
    if out.exists():
        print('Existing table preserved:',out)
    else:
        with out.open('x',newline='',encoding='utf-8') as f:
            writer=csv.DictWriter(f,fieldnames=allrows[0].keys())
            writer.writeheader();writer.writerows(allrows)
    return allrows


## Interpretation


## Optional sequential run


In [ ]:
AUTO_TRAIN = False
AUTO_PREDICT = True
AUTO_EVAL_SPLIT = "test"
AUTO_PRED_LIMIT = 0

from datetime import datetime, timezone
import subprocess

assert AUTO_EVAL_SPLIT in ("val", "test")
assert isinstance(AUTO_PRED_LIMIT, int) and AUTO_PRED_LIMIT >= 0
assert torch.cuda.is_available(), "GPU kernel unavailable"


job_id = os.environ.get("SLURM_JOB_ID")
print("SLURM_JOB_ID:", job_id or "not exposed in this kernel")
if job_id:
    try:
        proc = subprocess.run(
            ["squeue", "-j", job_id, "-h", "-o", "%i | state=%t | used=%M | limit=%l"],
            capture_output=True, text=True, timeout=10,
        )
        print("Job:", proc.stdout.strip() or proc.stderr.strip() or "not visible")
    except (OSError, subprocess.TimeoutExpired) as exc:
        print("Cannot read SLURM limit:", exc)

progress_log = EXP / "overnight_progress.jsonl"
def progress(stage, event, **extra):
    entry = {
        "timestamp_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "stage": stage, "event": event, **extra,
    }
    with progress_log.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(entry, ensure_ascii=False) + "\n")
    print(f"[{entry['timestamp_utc']}] {stage}: {event}")

def release_gpu():
    gc.collect()
    torch.cuda.empty_cache()

    try:
        torch.cuda.ipc_collect()
    except (RuntimeError, AttributeError):
        pass
    free, total = torch.cuda.mem_get_info()
    print(f"Free VRAM: {free / 1024**3:.2f}/{total / 1024**3:.2f} GiB")

if AUTO_TRAIN:
    for arch in ("A", "C"):
        stage = f"train_{arch}_clean"
        progress(stage, "start/resume")
        try:
            adapter_dir = train_clean(arch, full=True, resume=True)
            print("Adapter ready:", adapter_dir)
            progress(stage, "complete", adapter=str(adapter_dir))
        except Exception as exc:
            progress(stage, "errore", error=repr(exc))
            raise
        finally:


            release_gpu()

if AUTO_PREDICT:
    for arch, fine_tuned in (("A", False), ("A", True), ("C", False), ("C", True)):
        stage = f"predict_{arch}_{'ft' if fine_tuned else 'base'}_{AUTO_EVAL_SPLIT}"
        progress(stage, "start/resume")
        try:
            path = predict_clean(
                arch, fine_tuned=fine_tuned,
                split=AUTO_EVAL_SPLIT, limit=AUTO_PRED_LIMIT,
            )
            print("Predictions:", path)
            progress(stage, "complete", result=str(path))
        except Exception as exc:
            progress(stage, "errore", error=repr(exc))
            raise
        finally:
            release_gpu()

    progress("comparison", "start")
    summary = compare_clean(split=AUTO_EVAL_SPLIT, limit=AUTO_PRED_LIMIT)
    progress("comparison", "complete")
    print("\nEXPERIMENT COMPLETE:")
    for item in summary:
        print(item["name"], "Pair Accuracy =", round(item["pair_accuracy"], 4))

print("\nProgress log:", progress_log)
